# Analysis

Computed from the
saved predictions in `outputs/runs/`


In [ ]:
import sys
sys.path.insert(0, "..")

from math import comb

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch
from sklearn.metrics import roc_auc_score

from src.stats import (best_threshold_accuracy, bootstrap_ci, fmt, paired_diff,
                       siblings_in_training, two_sample_diff)

RUNS, TABLES, FIGS = "../outputs/runs", "../outputs/tables", "../outputs/figures"


def preds(run, ds="patras"):
    return pd.read_csv(f"{RUNS}/{ds}/{run}_preds.csv")


def folds(run, ds="patras"):
    return pd.read_csv(f"{RUNS}/{ds}/{run}_folds.csv").sort_values("fold").reset_index(drop=True)


def report(name, value, digits=3):
    print(f"{name:<55} {value:.{digits}f}")

## 3.1 Cohort

In [ ]:
report("N=36 against N=34, change in MCC", abs(folds("N36").mcc.mean() - folds("P1").mcc.mean()))

## 4.1 Subject-wise baseline (P1)

In [ ]:
p1 = folds("P1")
report("P1 accuracy %", 100 * p1.accuracy.mean(), 1)
m, lo, hi = bootstrap_ci(p1.mcc)
print(f"P1 MCC {m:.3f} [{lo:.3f}, {hi:.3f}]")
report("P1 sensitivity %", 100 * p1.sensitivity.mean(), 1)
report("P1 specificity %", 100 * p1.specificity.mean(), 1)
report("P1 ROC-AUC", p1.roc_auc.mean())

seeds = pd.DataFrame([folds(r)[["accuracy", "mcc", "roc_auc"]].mean()
                      for r in ["P1", "P1_s43", "P1_s44", "P1_s45"]], index=[42, 43, 44, 45])
print(seeds.round(3))
print(seeds.agg(["mean", "std"]).round(3))

report("P1 fold MCC, lowest", p1.mcc.min())
report("P1 fold MCC, highest", p1.mcc.max())
report("P1 folds classified perfectly", (p1.mcc == 1).sum(), 0)
report("P1 folds with one predicted class", (preds("P1").groupby("fold").prediction.nunique() == 1).sum(), 0)

d = preds("P1")
acc = [100 * ((d.probability >= t) == d.label).mean() for t in [0.3, 0.4, 0.5, 0.6, 0.7]]
print("accuracy at thresholds 0.3 to 0.7:", np.round(acc, 1))

## 4.2 Effect of the partition rule (P1 against P2)

In [ ]:
p2, d2 = folds("P2"), preds("P2")
report("P2 accuracy %", 100 * p2.accuracy.mean(), 1)
report("P2 MCC", p2.mcc.mean())
report("P2 errors", (d2.prediction != d2.label).sum(), 0)
report("P2 ROC-AUC", p2.roc_auc.mean())

# P2 folds are not participants, so the comparison with P1 is two-sample
print("P2 - P1 accuracy:", fmt(two_sample_diff(p2.accuracy, p1.accuracy), 100, 1))
print("P2 - P1 MCC:     ", fmt(two_sample_diff(p2.mcc, p1.mcc)))

print(f"mean best epoch: P1 {p1.best_epoch.mean():.1f}, P2 {p2.best_epoch.mean():.1f}")

print("siblings in training per P2 test frame:", siblings_in_training(d2).value_counts().to_dict())
report("P2 probabilities between 0.05 and 0.95", ((d2.probability > 0.05) & (d2.probability < 0.95)).sum(), 0)

Frame differences need the image files, so they are measured in
`04_baseline.ipynb` and only read here.

In [ ]:
print(pd.read_csv(f"{TABLES}/patras_frame_differences.csv").round(1).to_string(index=False))

report("P1 pooled ROC-AUC", roc_auc_score(d.label, d.probability))
report("P1 folds with best epoch >= 95", (p1.best_epoch >= 95).sum(), 0)

Controls: normalisation over all participants (P1G) and MobileNet-V2.

In [ ]:
g = folds("P1G")
report("P1G accuracy %", 100 * g.accuracy.mean(), 1)
report("P1G MCC", g.mcc.mean())
print("P1G - P1 MCC:     ", fmt(paired_diff(g.mcc, p1.mcc)))
print("P1G - P1 accuracy:", fmt(paired_diff(g.accuracy, p1.accuracy), 100, 1))

m1, m2 = folds("P1_mnv2"), folds("P2_mnv2")
report("MobileNet-V2 P1 accuracy %", 100 * m1.accuracy.mean(), 1)
report("MobileNet-V2 P1 MCC", m1.mcc.mean())
report("MobileNet-V2 P2 accuracy %", 100 * m2.accuracy.mean(), 1)
report("MobileNet-V2 P2 MCC", m2.mcc.mean())
print("MobileNet-V2 P2 - P1 accuracy:", fmt(two_sample_diff(m2.accuracy, m1.accuracy), 100, 1))

## Table 2 and Section 4.3 (frames or person)

In [ ]:
def rule_table(runs, ds="patras"):
    rows = []
    for rule, run in runs:
        f, d = folds(run, ds), preds(run, ds)
        rows.append(dict(rule=rule, MCC=f.mcc.mean(), Acc=100 * f.accuracy.mean(),
                         Sens=100 * f.sensitivity.mean(), Spec=100 * f.specificity.mean(),
                         Errors=f"{(d.prediction != d.label).sum()}/{len(d)}"))
    return pd.DataFrame(rows).round({"MCC": 3, "Acc": 1, "Sens": 1, "Spec": 1})


table2 = rule_table([("P1", "P1"), ("P3", "P3"), ("P4c", "P4c"), ("P4", "P4"), ("P2", "P2")])
table2.to_csv(f"{TABLES}/table2_patras_rules.csv", index=False)
print(table2.to_string(index=False))

In [ ]:
P4, P4c = folds("P4"), folds("P4c")
print("P4 - P4c sensitivity:", fmt(paired_diff(P4.sensitivity, P4c.sensitivity), 100, 1))
print("P4 - P4c specificity:", fmt(paired_diff(P4.specificity, P4c.specificity), 100, 1))
print("P4c - P1 sensitivity:", fmt(paired_diff(P4c.sensitivity, p1.sensitivity), 100, 1))
print("P4c - P1 specificity:", fmt(paired_diff(P4c.specificity, p1.specificity), 100, 1))

for run in ["P1", "P4"]:
    d = preds(run)
    report(f"sober frames called intoxicated, {run} (of 170)", ((d.label == 0) & (d.prediction == 1)).sum(), 0)

## 4.4 Why the effect is this large

In [ ]:
b, bf = preds("baseline"), folds("baseline")
print("feature chosen in each fold:", b.groupby("fold").feature.first().value_counts().to_dict())
report("baseline within-participant ROC-AUC", bf.roc_auc.mean())
report("baseline perfect rankings (of 34)", (bf.roc_auc == 1).sum(), 0)
report("perfect rankings expected by chance", 34 / comb(10, 5), 2)
report("baseline pooled ROC-AUC", roc_auc_score(b.label, b.probability))

v = pd.read_csv(f"{TABLES}/patras_p05_per_recording.csv")
rose = v.intoxicated > v.sober
cand = np.unique(np.r_[v.sober, v.intoxicated])
acc = [((v.sober < t).sum() + (v.intoxicated >= t).sum()) / (2 * len(v)) for t in cand]
t_best = cand[int(np.argmax(acc))]
wrong_sober, wrong_drunk = (v.sober >= t_best).sum(), (v.intoxicated < t_best).sum()
report("p05 rises after drinking (of 34)", rose.sum(), 0)
report("recordings wrong at the best single threshold (of 68)", wrong_sober + wrong_drunk, 0)

In [ ]:
for run, minutes in [("D1", 20), ("D2", 50), ("P1", 80)]:
    print(f"{minutes} min: within-participant ROC-AUC {folds(run).roc_auc.mean():.3f}")

## 4.5 Network against a one-feature baseline (Table 3)

In [ ]:
rows = []
for col, name in [("mcc", "MCC"), ("accuracy", "Acc"), ("roc_auc", "AUC")]:
    rows.append(dict(metric=name, resnet18=p1[col].mean(), p05=bf[col].mean(),
                     difference=fmt(paired_diff(p1[col], bf[col]))))
table3 = pd.DataFrame(rows)
table3.to_csv(f"{TABLES}/table3_baseline.csv", index=False)
print(table3.round(3).to_string(index=False))

d1 = preds("P1")
report("baseline accuracy at its best threshold %", 100 * best_threshold_accuracy(b.label, b.probability), 1)
report("network gain at its best threshold (points)",
       100 * (best_threshold_accuracy(d1.label, d1.probability) - (d1.prediction == d1.label).mean()), 1)

## 4.6 Second cohort (PUCV-DTF)

In [ ]:
u1, u2, ud2 = folds("P1", "pucv"), folds("P2", "pucv"), preds("P2", "pucv")
report("PUCV P1 accuracy %", 100 * u1.accuracy.mean(), 1)
m, lo, hi = bootstrap_ci(u1.mcc)
print(f"PUCV P1 MCC {m:.3f} [{lo:.3f}, {hi:.3f}]")
report("PUCV P1 ROC-AUC", u1.roc_auc.mean())
report("PUCV P2 errors", (ud2.prediction != ud2.label).sum(), 0)

print("PUCV P2 - P1 accuracy:", fmt(two_sample_diff(u2.accuracy, u1.accuracy), 100, 1))
print("PUCV P2 - P1 MCC:     ", fmt(two_sample_diff(u2.mcc, u1.mcc)))

print("siblings in training per P2 test frame:", siblings_in_training(ud2).value_counts().to_dict())

In [ ]:
table4 = rule_table([("P1", "P1"), ("P3", "P3"), ("P4c", "P4c"), ("P4", "P4"), ("P2", "P2")], "pucv")
table4.to_csv(f"{TABLES}/table4_pucv_rules.csv", index=False)
print(table4.to_string(index=False))

In [ ]:
u3, u4, u4c = folds("P3", "pucv"), folds("P4", "pucv"), folds("P4c", "pucv")

# P3 folds are not participants, so the comparison with P1 is two-sample
print("PUCV P3 - P1 accuracy:", fmt(two_sample_diff(u3.accuracy, u1.accuracy), 100, 1))
print("PUCV P3 MCC", fmt(bootstrap_ci(u3.mcc)))

print("PUCV P4 - P4c sensitivity:", fmt(paired_diff(u4.sensitivity, u4c.sensitivity), 100, 1))
print("PUCV P4 - P4c specificity:", fmt(paired_diff(u4.specificity, u4c.specificity), 100, 1))
print("PUCV P4c - P1 sensitivity:", fmt(paired_diff(u4c.sensitivity, u1.sensitivity), 100, 1))
print("PUCV P4c - P1 specificity:", fmt(paired_diff(u4c.specificity, u1.specificity), 100, 1))

for run in ["P1", "P4"]:
    d = preds(run, "pucv")
    report(f"PUCV sober frames called intoxicated, {run} (of 230)", ((d.label == 0) & (d.prediction == 1)).sum(), 0)

In [ ]:
ub, ubf, ud1 = preds("baseline", "pucv"), folds("baseline", "pucv"), preds("P1", "pucv")
print("feature chosen in each fold:", ub.groupby("fold").feature.first().value_counts().to_dict())
report("PUCV baseline perfect rankings (of 46)", (ubf.roc_auc == 1).sum(), 0)
report("PUCV perfect rankings expected by chance", 46 / comb(10, 5), 2)
report("PUCV baseline within-participant ROC-AUC", ubf.roc_auc.mean())
report("PUCV baseline pooled ROC-AUC", roc_auc_score(ub.label, ub.probability))
report("PUCV network within-participant ROC-AUC", u1.roc_auc.mean())
report("PUCV network pooled ROC-AUC", roc_auc_score(ud1.label, ud1.probability))

w = pd.read_csv(f"{TABLES}/pucv_p50_per_recording.csv")
report("PUCV p50 rises after drinking (of 46)", (w.intoxicated > w.sober).sum(), 0)

print("PUCV network - baseline ROC-AUC: ", fmt(paired_diff(u1.roc_auc, ubf.roc_auc)))
print("PUCV network - baseline MCC:     ", fmt(paired_diff(u1.mcc, ubf.mcc)))
print("PUCV network - baseline accuracy:", fmt(paired_diff(u1.accuracy, ubf.accuracy), 100, 1))

## Figures

In [ ]:
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.size": 9})
acc = {r.rule: f"{r.Acc:.1f}" for r in table2.itertuples()}

ROWS = [("P1", "subject $k$", "5 sober + 5 intoxicated", "nothing", ""),
        ("P3", "2 whole recordings", "from 2 participants", "their other recording", "carrying the opposite label"),
        ("P4c", "subject $k$", "same as P1", "nothing (control)", "10 frames of another participant"),
        ("P4", "subject $k$", "same as P1", "10 of their own frames", "from the 20 and 50 min sessions"),
        ("P2", "10 random frames", "from the pooled set", "siblings of every test frame", "2 to 4 per frame, same burst")]
H, GAP, TOP = 0.62, 0.78, 4.0

plt.rcParams["font.family"] = "DejaVu Sans"
fig, ax = plt.subplots(figsize=(6.4, 2.0))
ax.set_xlim(0, 10)
ax.set_ylim(0.72, 5.02)
ax.axis("off")
for x, label in [(0.85, "test set"), (4.05, "what training holds about that participant"), (9.0, "acc %")]:
    ax.text(x, 4.76, label, fontsize=7, style="italic", color="0.35")


def box(x, w, y, top, sub):
    ax.add_patch(FancyBboxPatch((x, y), w, H, boxstyle="round,pad=0.015,rounding_size=0.07",
                                lw=0.7, ec="0.35", fc="white"))
    ax.text(x + w / 2, y + H * (0.68 if sub else 0.5), top, ha="center", va="center", fontsize=8)
    if sub:
        ax.text(x + w / 2, y + H * 0.24, sub, ha="center", va="center", fontsize=6.3, color="0.4")


for i, (rule, a1, a2, b1, b2) in enumerate(ROWS):
    y = TOP - i * GAP
    ax.text(0.05, y + H / 2, rule, fontsize=8.5, fontweight="bold", va="center")
    box(0.85, 3.05, y, a1, a2)
    box(4.05, 4.55, y, b1, b2)
    ax.text(9.0, y + H / 2, acc[rule], fontsize=8.5, fontweight="bold", va="center")

fig.savefig(f"{FIGS}/Figure_1.pdf", bbox_inches="tight")
fig.savefig(f"{FIGS}/Figure_1.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.rcParams["font.family"] = "DejaVu Sans"
sober, drunk = v.sober.to_numpy(), v.intoxicated.to_numpy()
n, fell = len(v), ~rose.to_numpy()
order = np.argsort(sober)

fig = plt.figure(figsize=(6.4, 2.7))
gs = GridSpec(1, 2, width_ratios=[1.7, 1], wspace=0.42)

ax = fig.add_subplot(gs[0])
ax.axhline(t_best, color="0.25", ls=":", lw=1.1, zorder=0)
for i, k in enumerate(order):
    ax.plot([i, i], [sober[k], drunk[k]], color="crimson" if fell[k] else "0.4",
            ls="--" if fell[k] else "-", lw=1.5 if fell[k] else 1.0, zorder=1)
    ax.scatter(i, sober[k], s=15, facecolor="white", edgecolor="0.2", lw=0.8, zorder=3)
    ax.scatter(i, drunk[k], s=15, color="0.2", zorder=3)
    if fell[k]:
        ax.scatter([i, i], [sober[k], drunk[k]], s=40, marker="x", color="crimson", lw=1.3, zorder=4)
lo, hi = sober.min(), max(sober.max(), drunk.max())
ax.annotate("best single threshold", xy=(n - 1, t_best), xytext=(n * 0.60, lo + 0.04 * (hi - lo)),
            fontsize=7, color="0.3", annotation_clip=False,
            arrowprops=dict(arrowstyle="-", color="0.55", lw=0.6, shrinkA=0, shrinkB=2))
ax.set_xlabel("participant, sorted by sober level")
ax.set_ylabel("periorbital p05")
ax.set_title(f"(a) within a participant: {int(rose.sum())} of {n} rise")
ax.set_xticks([])

ax2 = fig.add_subplot(gs[1])
ax2.axvline(t_best, color="0.25", ls=":", lw=1.1, zorder=0)
jitter = np.random.default_rng(1)
for vals, y0, filled in [(sober, 1.0, False), (drunk, 0.0, True)]:
    j = jitter.uniform(-0.13, 0.13, len(vals))
    wrong = vals < t_best if filled else vals >= t_best
    ax2.scatter(vals, y0 + j, s=16, facecolor="0.2" if filled else "white", edgecolor="0.2", lw=0.8, zorder=2)
    ax2.scatter(vals[wrong], y0 + j[wrong], s=38, marker="x", color="crimson", lw=1.1, zorder=3)
ax2.set_yticks([1.0, 0.0])
ax2.set_yticklabels(["sober", "intoxicated"])
ax2.set_ylim(-0.45, 1.45)
ax2.margins(x=0.08)
ax2.set_xlabel("periorbital p05")
ax2.set_title(f"(b) pooled: {wrong_sober + wrong_drunk} of {2 * n} wrong")
ax2.text(0.98, 0.96, f"{wrong_sober} sober above", transform=ax2.transAxes, fontsize=6.5,
         color="crimson", ha="right", va="top")
ax2.text(0.02, 0.04, f"{wrong_drunk} intoxicated below", transform=ax2.transAxes, fontsize=6.5,
         color="crimson", ha="left", va="bottom")
ax2.spines[["top", "right"]].set_visible(False)

fig.savefig(f"{FIGS}/Figure_2.pdf", bbox_inches="tight")
fig.savefig(f"{FIGS}/Figure_2.png", dpi=300, bbox_inches="tight")
plt.show()